# Analysis II - Models and controlled experiments
**Evidence classes:** The first plots here are counterfactual calculations using the observed event timestamps from [Analysis I](01_observed_workloads.ipynb). They are not measured packets, response times, watts, or joules. A separate experimental results table is defined below; it remains empty until comparable service runs have been performed.

## Questions
1. Which batch size and maximum wait might reduce application sends while holding events briefly? Apply an explicitly defined count-or-timeout model to a fixed replay.
2. Which settings actually reduce bytes and joules per successful task without breaking delivery or latency targets? Answer with repeated, randomized-order measurements on the same testbed.
3. Which settings are nondominated? Compute a Pareto frontier only after correctness and service-level thresholds have been enforced.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display, Markdown
PATH = Path('../data/raw/intel/data.txt.gz')
assert PATH.exists(), 'Run Analysis I with the downloaded original Intel archive first.'
raw = pd.read_csv(PATH,sep=' ',names=['date','time','epoch','moteid','temperature','humidity','light','voltage'],header=None,compression='gzip',low_memory=False)
times = pd.to_datetime(raw.date.astype(str)+' '+raw.time.astype(str),errors='coerce')
ids = pd.to_numeric(raw.moteid,errors='coerce')
events = times[times.notna() & ids.between(1,54)].sort_values(kind='stable')
seconds = events.astype('int64').to_numpy()/1e9
print(f'Fixed replay: {len(seconds):,} timestamped, roster-valid events.')


Fixed replay: 2,303,286 timestamped, roster-valid events.


## Explicit batch model
Within a single logical aggregation stream, flush when the batch reaches its record limit or at the first event's deadline, whichever comes first. An event arriving on or after the deadline begins the next batch. Flush the final partial batch at its deadline. Holding time is an **added waiting component**, not measured end-to-end delivery latency. The model assumes perfect delivery and ignores bytes, packetization, headers, retries, acknowledgements, CPU, and energy.


In [2]:
def model_batches(times, max_count, max_wait_s):
    if max_count < 1 or max_wait_s < 0: raise ValueError('Invalid settings')
    if len(times) == 0: return 0,np.array([])
    waits = np.empty(len(times),dtype=float); start = 0; batches = 0
    for i in range(len(times)):
        if i > start and times[i] >= times[start]+max_wait_s:
            waits[start:i] = times[start]+max_wait_s-times[start:i]
            batches += 1; start = i
        if i-start+1 == max_count:
            waits[start:i+1] = times[i]-times[start:i+1]
            batches += 1; start = i+1
    if start < len(times):
        waits[start:] = times[start]+max_wait_s-times[start:]
        batches += 1
    return batches,waits

# Cases chosen so flush boundaries and waiting times are hand-checkable.
assert model_batches(np.array([0.,2.,10.]),8,5)[0] == 2
np.testing.assert_allclose(model_batches(np.array([0.,2.,10.]),8,5)[1],[5.,3.,5.])
np.testing.assert_allclose(model_batches(np.array([0.,2.,10.]),1,60)[1],[0.,0.,0.])
print('Two boundary checks passed.')


Two boundary checks passed.


In [3]:
rows=[]
for count in (1,8,32):
    for wait in (1,10,60):
        batches,holds = model_batches(seconds,count,wait)
        rows.append({'max_count':count,'max_wait_s':wait,'modelled_batches':batches,'modelled_reduction_pct':100*(1-batches/len(seconds)),'modelled_p95_hold_s':float(np.quantile(holds,.95))})
scenarios=pd.DataFrame(rows)
display(scenarios)
fig=px.line(scenarios,x='modelled_p95_hold_s',y='modelled_batches',color='max_count',symbol='max_wait_s',markers=True,hover_data=['modelled_reduction_pct'],title='6. Modelled sends versus modelled holding time; no energy measurement'); fig.show()
fig=px.bar(scenarios,x='max_count',y='modelled_reduction_pct',color='max_wait_s',barmode='group',title='7. Modelled reduction relative to one application send per event'); fig.show()
display(Markdown('**Technical:** These are outputs of a deterministic batching rule conditional on logged timestamps. They are not an estimate of packet, byte, CPU or energy savings; tightly clustered timestamps can make modelled holding time appear very small.  \n**Plain language:** Combining readings may mean fewer sends, but we still need to run the actual server to learn whether it saves electricity and meets delivery deadlines.'))


,max_count,max_wait_s,modelled_batches,modelled_reduction_pct,modelled_p95_hold_s
0,1,1,2303286,0.000000,0.000000
1,1,10,2303286,0.000000,0.000000
2,1,60,2303286,0.000000,0.000000
3,8,1,796336,65.426091,1.000000
4,8,10,335460,85.435591,9.803994
5,8,60,288065,87.493303,22.230138
6,32,1,779894,66.139941,1.000000
7,32,10,169841,92.626144,10.000000
8,32,60,75758,96.710873,39.075037


**Technical:** These are outputs of a deterministic batching rule conditional on logged timestamps. They are not an estimate of packet, byte, CPU or energy savings; tightly clustered timestamps can make modelled holding time appear very small.  
**Plain language:** Combining readings may mean fewer sends, but we still need to run the actual server to learn whether it saves electricity and meets delivery deadlines.

## Sensitivity to aggregation scope

The first counterfactual above treats all 54 sensors as one logical stream. That is an optimistic upper bound if records must preserve a route, tenant, schema, ordering key, or destination. Recompute the exact same count-or-timeout rule separately for each sensor, which is an illustrative conservative boundary. Sensor identity is not a real service routing key, so neither boundary predicts a deployable result.

In [4]:
by_mote = [g.timestamp.astype('int64').to_numpy()/1e9
           for _, g in raw.loc[times.notna() & ids.between(1,54), ['moteid']].assign(timestamp=times).groupby('moteid')]
scoped=[]
for count in (1,8,32):
    for wait in (1,10,60):
        parts=[model_batches(t,count,wait) for t in by_mote]
        n=sum(part[0] for part in parts)
        holds=np.concatenate([part[1] for part in parts])
        scoped.append({'max_count':count,'max_wait_s':wait,'per_mote_batches':n,
                       'per_mote_reduction_pct':100*(1-n/len(seconds)),
                       'per_mote_p95_hold_s':float(np.quantile(holds,.95))})
scoped=pd.DataFrame(scoped)
comparison=scenarios.merge(scoped,on=['max_count','max_wait_s'])
display(comparison[['max_count','max_wait_s','modelled_reduction_pct','per_mote_reduction_pct',
                    'modelled_p95_hold_s','per_mote_p95_hold_s']])
assert (comparison.per_mote_batches >= comparison.modelled_batches).all()

,max_count,max_wait_s,modelled_reduction_pct,per_mote_reduction_pct,modelled_p95_hold_s,per_mote_p95_hold_s
0,1,1,0.000000,0.000000,0.000000,0.000000e+00
1,1,10,0.000000,0.000000,0.000000,0.000000e+00
2,1,60,0.000000,0.000000,0.000000,0.000000e+00
3,8,1,65.426091,21.972087,1.000000,1.094252e+06
4,8,10,85.435591,22.182438,9.803994,1.094261e+06
5,8,60,87.493303,59.642658,22.230138,1.094300e+06
6,32,1,66.139941,26.671069,1.000000,1.094255e+06
7,32,10,92.626144,26.874604,10.000000,1.094264e+06
8,32,60,96.710873,62.132536,39.075037,1.094307e+06


### Observations and argument

At 8 records and a 10-second deadline, pooling all accepted events yields 335,460 modelled sends (85.44% below one send per event) and p95 hold of 9.80 s. The per-sensor boundary yields 2,294,458 sends (0.383% below baseline) and p95 hold of 10 s. At a 60-second deadline, the per-sensor boundary reaches 48.98% modelled send reduction, while p95 hold reaches the full 60 s. Raising the per-sensor count cap from 8 to 32 barely changes the result at 10 or 60 seconds because timeout, rather than count, determines most flushes.

**Deduction:** The apparent optimum is dominated by the *compatibility domain* and deadline, not simply batch size. A model that combines independent streams can manufacture a large apparent gain and an incorrect hold-time conclusion. The optimiser must first encode which records can legally share a request, preserving ordering, destination, payload limits, and delivery semantics. It should then search configuration settings under explicit latency and correctness constraints. These are modelled application sends and holds; no packet, byte, watt, or joule saving has been observed.

## Experimental protocol and result schema

The [controlled study contract](../experiment/contract.json) fixes topology, payload, compatibility, acknowledgement and illustrative limits. The [runner](../experiment/README.md) creates a randomized plan **before** execution and writes one JSON result for every planned run, including correctness, end-to-end latency, throughput, application body bytes, CPU time and optional packet counters. A failed or missing run stays in the plan. [Selection](05_select_configs.py) requires every repeat to pass and reports run-level variation; it computes a joules/latency Pareto frontier only after one consistent AC-wall or battery-DC power series are integrated into **all** runs. See the executable section below for imported results.

## Interpretation boundary
A smaller count of application sends is a hypothesis about work avoided. A lower measured energy per *successful* task, with comparable reliability and latency, supports a testbed-specific sustainability claim. It does not automatically apply to other devices, workloads or facilities. See the [research plan](research-plan.md) and [assessment](assessment.md) for the next data collection steps.


## Reassessment for an actual optimiser

1. **Define the decision surface.** Map real producer-to-consumer edges, per-route payloads, ordering keys, fan-out, and protocol settings. The Intel log supplies none of these. The pooled-versus-per-sensor contrast shows why compatibility must be a first-class constraint.
2. **Measure a complete task.** Use identical input IDs and a verified database outcome. Count request and response bytes, packetization, retries, queue depth, CPU, and declared system joules. Preserve delivery guarantees. A send-count reduction is only a candidate mechanism.
3. **Search under hard constraints.** Predeclare per-route p95/p99 latency, throughput, loss and duplicate limits, memory cap, and payload size. Include baseline/no batching and single-variable ablations. Reject any candidate that fails any repeated run; compare feasible candidates on measured joules per successful event and latency.
4. **Account for operating regimes.** Characterize quiet, steady and bursty periods using availability-aware windows and a real microservice trace. Evaluate separate frontiers by route, workload, hardware and utilization. Include idle power, since a smaller busy component may have little whole-host effect.
5. **Make adaptation safe.** If the eventual optimiser changes settings online, specify decision interval, telemetry cost, hysteresis, rollback, and a conservative fallback. Validate it against fixed settings on held-out workloads and report uncertainty, including adverse results.

The immediate project claim should be **a constrained, empirically evaluated communication-configuration optimiser for one instrumented service path**. General cross-protocol or universal homelab energy claims require evidence across more paths and machines. No measured experiment file is present in this repository, so the Pareto and energy conclusions remain open.

## Implemented controlled service path

The [study contract](../experiment/contract.json) now fixes a producer → HTTP → bounded queue → one worker → PostgreSQL path. Batches must share route, tenant and schema; the producer assigns a per-route sequence. HTTP 200 follows database commit, retries keep the same event IDs, and a primary key makes completed IDs idempotent. The predeclared illustrative limits are p95 delivery ≤ 2 s, p99 ≤ 5 s, zero missing/extra/duplicate completed IDs or send errors, and throughput at least 90% of the same-profile baseline median. Configurations `1:0`, `8:100 ms`, and `32:500 ms` are randomized over three repeats per profile with a recorded seed. Source hours are accelerated 120-fold for this stress replay.

The Lenovo Yoga 7i is the preferred primary hardware target. SQLite functional checks and a ZOTAC PostgreSQL 100-event smoke run verified completed IDs, but the ZOTAC matrix was stopped at the owner’s request before completion and is excluded from selection. Complete repeated results are read below when a Yoga matrix is present. Runner interface counters can measure producer-edge packets and bytes on Linux containers; per-edge app-to-database capture and whole-host watts remain separate instrumentation work. Any energy Pareto remains absent until same-boundary system-power samples cover every run.

In [5]:
from importlib.util import spec_from_file_location, module_from_spec
plans=sorted(Path('results/experimental_runs').glob('*-plan.json')) if Path('results/experimental_runs').exists() else []
if plans:
    spec=spec_from_file_location('study_selection','05_select_configs.py')
    selection=module_from_spec(spec); spec.loader.exec_module(selection)
    study_summary=selection.summarize(plans[-1],Path('results/selection'))
    display(study_summary)
else:
    display(Markdown('**Controlled study status:** no repeated-run plan has been imported yet. The testbed and replay manifests are ready; energy remains unmeasured.'))

**Controlled study status:** no repeated-run plan has been imported yet. The testbed and replay manifests are ready; energy remains unmeasured.

## Safe adaptation and generalization status

The [controller](../experiment/controller.py) defines telemetry freshness, a conservative system-energy benefit requirement, switching-cost evidence, two stable observations, minimum dwell, and immediate rollback on loss, queue growth, stale telemetry or latency breach. With no whole-host energy measurements it deliberately stays at the no-batching baseline. This is a safety scaffold; its monitoring overhead, switching cost, and behavior on held-out workloads still require measured trials.

The Yoga is the preferred first hardware profile; a Windows power-counter probe and external watt-meter integration path are ready. The same pinned multi-architecture images and replay manifest are ready for the M1 MacBook Air, but that machine is not accessible from this checkout. Hardware frontiers must be kept separate. CPU package counters, macOS subsystem estimates, application sends, and body bytes cannot be substituted for declared system joules.